# Task-02 — YOLOv8s training on Google Colab

This notebook trains the **Drone Human Detection & Counting System** detector
on a Colab GPU (T4 / L4 / A100). It is the cloud-side companion to the
local repo at `Drone-Human-Detection-Counting-System/`:

* **Local (PC)** — Task-01 data cleaning, YOLO dataset layout build,
  sample inference, counting overlays, evaluation.
* **Colab (this notebook)** — Task-02 training. Produces `best.pt`,
  `results.png`, `confusion_matrix.png`, etc.

## Workflow

1. Run `python src\data\build_yolo_dataset.py ...` locally to produce
   `data/processed/visdrone/{train,val,test}/{images,labels}`.
2. Zip the prepared dataset and upload it to your Google Drive at:
   `MyDrive/drone-detection/visdrone_yolo.zip`.
3. Run this notebook end-to-end. The trained weights and Ultralytics
   plots get saved to: `MyDrive/drone-detection/runs/yolov8s_visdrone/`.
4. Download `best.pt` (and `results.png`, `confusion_matrix.png`) and
   drop them into `outputs/weights/` / `outputs/figures/task02/` in the
   local repo. Then run inference / counting / evaluation locally.

## Required Colab runtime

* GPU enabled (`Runtime → Change runtime type → T4 / L4 / A100`).
* ~25 GB Drive free if you keep checkpoints + plots.

## 1. Sanity check the runtime
Confirm we got a GPU and which one.

In [ ]:
!nvidia-smi | head -n 20

## 2. Install dependencies
Ultralytics pulls a compatible PyTorch wheel automatically on Colab.

In [ ]:
%pip install -q "ultralytics>=8.3" pyyaml

## 3. Mount Google Drive and unzip the dataset

Make sure `visdrone_yolo.zip` exists at:
`MyDrive/drone-detection/visdrone_yolo.zip`

The zip is expected to contain:

```text
train/images/*.jpg
train/labels/*.txt
val/images/*.jpg
val/labels/*.txt
test/images/*.jpg
test/labels/*.txt
```

(`labels_yolo/` mirrors are fine but unused here.)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil, time
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/drone-detection")
ZIP_PATH = DRIVE_ROOT / "visdrone_yolo.zip"
DATA_ROOT = Path("/content/visdrone_yolo")
RUNS_ROOT_DRIVE = DRIVE_ROOT / "runs"
RUNS_ROOT_DRIVE.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f"Dataset zip not found at {ZIP_PATH}. Upload it to Drive first."
print(f"Found dataset zip: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e9:.2f} GB)")

if DATA_ROOT.exists():
    print(f"{DATA_ROOT} already exists, skipping unzip")
else:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    !unzip -q "{ZIP_PATH}" -d "{DATA_ROOT}"
    print(f"Unzip took {time.time() - t0:.1f}s")

for split in ("train", "val", "test"):
    img_dir = DATA_ROOT / split / "images"
    lab_dir = DATA_ROOT / split / "labels"
    assert img_dir.exists(), f"missing {img_dir}"
    assert lab_dir.exists(), f"missing {lab_dir}"
    print(f"[{split}] images={len(list(img_dir.glob('*.jpg')))}  labels={len(list(lab_dir.glob('*.txt')))}")

## 4. Write the Ultralytics data YAML

This points YOLOv8 at the unzipped folders.

In [ ]:
import yaml

DATA_YAML = Path("/content/task02_data.yaml")
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATA_ROOT),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {0: "person", 1: "car"},
    "nc": 2,
}, sort_keys=False))
print(DATA_YAML.read_text())

## 5. Train YOLOv8s

Default Colab T4-friendly hyperparameters. Edit if you got an L4 / A100
and want to crank `imgsz` / `batch` higher.

* `imgsz=800` — bigger than the default 640 to better catch tiny aerial
  pedestrians; T4 / L4 can fit it at `batch=16` in AMP.
* `epochs=40` — long enough to clearly see convergence and to trigger
  `close_mosaic=10` in the last few epochs.
* `mosaic=1.0` then closed for the last 10 epochs (standard YOLOv8
  recipe — helps generalization early, clean targets late).
* `degrees=0.0`, `flipud=0.0` — aerial footage has a consistent up
  direction; rotating/upside-down flipping hurts.
* Mixup / copy-paste off — kept small-object semantics intact.

In [ ]:
from ultralytics import YOLO

PROJECT_DIR = Path("/content/runs/task02")
RUN_NAME = "yolov8s_visdrone"

model = YOLO("yolov8s.pt")
results = model.train(
    data=str(DATA_YAML),
    epochs=40,
    imgsz=800,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    cos_lr=True,
    patience=20,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.0,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    mosaic=1.0,
    close_mosaic=10,
    mixup=0.0,
    erasing=0.0,
    cache=False,
    workers=4,
    seed=42,
    deterministic=True,
    amp=True,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    save=True,
    verbose=True,
)

save_dir = Path(results.save_dir)
print("Training complete. Run dir:", save_dir)

## 6. Final validation pass on `best.pt`

Runs at the same image size used during training and emits the standard
PR / confusion-matrix plots.

In [ ]:
import json

best_weights = save_dir / "weights" / "best.pt"
print("best.pt:", best_weights, "exists:", best_weights.exists())

val_model = YOLO(str(best_weights))
val_results = val_model.val(
    data=str(DATA_YAML),
    imgsz=800,
    batch=16,
    plots=True,
    split="val",
)

metrics = {k: float(v) for k, v in val_results.results_dict.items()
           if isinstance(v, (int, float))}
print(json.dumps(metrics, indent=2))

## 7. (Optional) evaluation on the test-dev split

VisDrone's `test-dev` split has public labels. Running val on it gives a
second, larger evaluation set independent of the per-epoch val.

In [ ]:
test_results = val_model.val(
    data=str(DATA_YAML),
    imgsz=800,
    batch=16,
    plots=True,
    split="test",
)
test_metrics = {k: float(v) for k, v in test_results.results_dict.items()
                if isinstance(v, (int, float))}
print(json.dumps(test_metrics, indent=2))

## 8. Persist results back to Drive

Copies the whole Ultralytics run directory (weights, plots, results.csv,
args.yaml) to Drive so you don't lose it when the Colab session dies.
A tiny `task02_training_summary.json` file is written alongside `best.pt`
to make it easy to embed metrics into the repo's README.

In [ ]:
import datetime, torch

drive_run_dir = RUNS_ROOT_DRIVE / save_dir.name
if drive_run_dir.exists():
    shutil.rmtree(drive_run_dir)
shutil.copytree(save_dir, drive_run_dir)
print("Copied run dir to:", drive_run_dir)

summary = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "model": "yolov8s.pt",
    "data": str(DATA_YAML),
    "imgsz": 800,
    "batch": 16,
    "epochs": 40,
    "val_metrics": metrics,
    "test_metrics": test_metrics if "test_metrics" in globals() else None,
    "best_weights_in_drive": str(drive_run_dir / "weights" / "best.pt"),
    "results_png_in_drive": str(drive_run_dir / "results.png"),
    "confusion_matrix_in_drive": str(drive_run_dir / "confusion_matrix.png"),
}
(drive_run_dir / "task02_training_summary.json").write_text(
    json.dumps(summary, indent=2)
)
print("Wrote training summary:", drive_run_dir / "task02_training_summary.json")

## 9. (Optional) sample qualitative predictions from Colab

Quick sanity-check rendering. The real inference / counting overlays are
generated locally in the repo via `src/infer/infer_images.py` so they
live inside the deliverable artifact directory.

In [ ]:
import random, glob
from IPython.display import Image, display

val_images = sorted(glob.glob(str(DATA_ROOT / "val" / "images" / "*.jpg")))
random.seed(0)
samples = random.sample(val_images, k=6)

sample_dir = Path("/content/sample_predictions")
if sample_dir.exists():
    shutil.rmtree(sample_dir)
sample_dir.mkdir(parents=True)

val_model.predict(
    source=samples,
    imgsz=800,
    conf=0.25,
    iou=0.5,
    save=True,
    project=str(sample_dir.parent),
    name=sample_dir.name,
    exist_ok=True,
)

rendered = sorted(Path(sample_dir).rglob("*.jpg"))
for p in rendered:
    display(Image(filename=str(p), width=720))

## 10. What to copy back into the local repo

After the notebook finishes, download these files from
`MyDrive/drone-detection/runs/yolov8s_visdrone/` and drop them into the
local repo so the deliverable contains real artifacts:

| Drive file                        | Local destination                                       |
| --------------------------------- | ------------------------------------------------------- |
| `weights/best.pt`                 | `outputs/weights/best.pt`                               |
| `results.png`                     | `outputs/figures/task02/results.png`                    |
| `results.csv`                     | `outputs/metrics/task02_results.csv`                    |
| `confusion_matrix.png`            | `outputs/figures/task02/confusion_matrix.png`           |
| `confusion_matrix_normalized.png` | `outputs/figures/task02/confusion_matrix_normalized.png`|
| `PR_curve.png`                    | `outputs/figures/task02/PR_curve.png`                   |
| `task02_training_summary.json`    | `outputs/metrics/task02_training_summary.json`          |

Then on the local machine run:

```powershell
python src\eval\evaluate_detector.py --weights outputs\weights\best.pt
python src\infer\infer_images.py --weights outputs\weights\best.pt
```